# 11. VAAS v2 Quick Start

This notebook demonstrates the fastest way to start using **VAAS v2** for image anomaly detection and localisation.

It covers:

- loading a VAAS v2 model from Hugging Face
- running inference on a single image
- reading `S_F`, `S_P`, and `S_H`
- generating a visual explanation
- briefly comparing v2 against v1 on the same image

## 1. Install dependencies

This notebook installs `vaas` and the runtime dependencies required for inference.

In [ ]:
!pip install -q vaas torch torchvision

## 2. Imports

In [ ]:
from io import BytesIO

import requests
from PIL import Image
from IPython.display import Image as IPImage, display

from vaas.inference.pipeline import VAASPipeline

## 3. Load an example image

This example uses a public image from the VAAS repository.

In [ ]:
url = "https://raw.githubusercontent.com/OBA-Research/VAAS/main/examples/images/COCO_DF_C110B00000_00539519.jpg"
image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")
image

## 4. Load the VAAS v2 pipeline

You can switch between `v2-base-df2023`, `v2-medium-df2023`, and `v2-large-df2023`.

In [ ]:
pipeline_v2 = VAASPipeline.from_pretrained(
    repo_id="OBA-Research/vaas",
    model_variant="v2-base-df2023",
    device="cpu",   # change to "cuda" if a GPU runtime is enabled
    alpha=0.5,
)

## 5. Run inference

The pipeline returns three scores and a dense anomaly map.

In [ ]:
result_v2 = pipeline_v2(image)

print("VAAS v2 output")
print("S_F:", result_v2["S_F"])
print("S_P:", result_v2["S_P"])
print("S_H:", result_v2["S_H"])
print("Anomaly map shape:", result_v2["anomaly_map"].shape)

## 6. Interpret the scores

- `S_F` measures global attention-based anomaly
- `S_P` measures patch-level anomaly
- `S_H` is the hybrid anomaly score

Higher values indicate stronger anomaly evidence.

## 7. Generate a visual explanation

This produces a figure containing:

- original image
- patch-level heatmap
- binary anomaly mask
- Fx attention map
- hybrid score gauge

In [ ]:
pipeline_v2.visualize(
    image=image,
    save_path="vaas_v2_quick_start.png",
    mode="all",
    threshold=0.5,
)

display(IPImage("vaas_v2_quick_start.png"))

## 8. Inspect detailed output

`forward_detailed` exposes the standard scores plus model metadata.

In [ ]:
detailed_v2 = pipeline_v2.forward_detailed(image)

print("Variant:", detailed_v2["variant"])
print("Metadata keys:", sorted(detailed_v2["metadata"].keys()))

## 9. Brief v1 vs v2 comparison

This is a light comparison on the same image using one v1 and one v2 model variant.

In [ ]:
pipeline_v1 = VAASPipeline.from_pretrained(
    repo_id="OBA-Research/vaas",
    model_variant="v1-base-df2023",
    device="cpu",
    alpha=0.5,
)

result_v1 = pipeline_v1(image)

print("v1 S_F:", result_v1["S_F"])
print("v1 S_P:", result_v1["S_P"])
print("v1 S_H:", result_v1["S_H"])
print()
print("v2 S_F:", result_v2["S_F"])
print("v2 S_P:", result_v2["S_P"])
print("v2 S_H:", result_v2["S_H"])

## 10. Save the anomaly map if needed

This can be useful for downstream analysis or reporting.

In [ ]:
import numpy as np

np.save("vaas_v2_anomaly_map.npy", result_v2["anomaly_map"])
print("Saved anomaly map to vaas_v2_anomaly_map.npy")

## 11. Summary

In this notebook you:

- loaded a VAAS v2 model
- ran single-image inference
- generated a visual explanation
- inspected metadata through `forward_detailed`
- compared v2 against v1 on the same image

Next notebook: [**VAAS v2 model variants comparison**](https://colab.research.google.com/drive/1UU-B4nc4ML3IEy0SDKi2aT9sorG0Z9eF?usp=sharing)